# Causal Trace: Standard

Runs the existing standard-style trace through the shared prototype runner and plots per-layer indirect effects.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import hydra
import matplotlib.pyplot as plt
import pandas as pd
from src.causal_trace.prototype import run_standard_trace

MODEL = 'gpt2-large'
NUM_PROMPTS = 1
NUM_NOISE = 10


In [ ]:
with hydra.initialize_config_dir(config_dir=str(ROOT / 'src' / 'config'), version_base=None):
    cfg = hydra.compose(
        config_name='latium',
        overrides=[
            'command=causal_trace',
            f'model={MODEL}',
            f'generation.num_of_runs={NUM_PROMPTS}',
            f'tracing.num_noise_samples={NUM_NOISE}',
        ],
    )

out_dir = Path(run_standard_trace(cfg))
out_dir


In [ ]:
scores = pd.read_csv(out_dir / 'wide_layer_scores.csv')
layer_cols = [col for col in scores.columns if col.startswith('layer_')]
display(scores[['prompt_id', 'subject', 'target', 'trace_reliable', 'candidate_layers']])

ax = scores[layer_cols].T.plot(figsize=(10, 4), legend=False, marker='o')
ax.set_xlabel('Layer')
ax.set_ylabel('Mean indirect effect')
ax.set_title('Standard trace layer effects')
plt.show()
